In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

OUT = Path('../data_dan/forest_change_map.html')

In [ ]:
# Load data
diff = pd.read_csv('../data_dan/MAIN_1990-2025forest_difference.csv')[['Entity', 'Code', 'Share of land covered by forest']]
diff.columns = ['Entity', 'Code', 'Forest_change']

ekc = pd.read_csv('../data_dan/forest_ekc_model.csv')[['Code', 'Forest_1990', 'Forest_2025']]
data = diff.merge(ekc, on='Code', how='left')

# Round for display in popup
data['Forest_change_r'] = data['Forest_change'].round(2)
data['Forest_1990_r']   = data['Forest_1990'].round(2)
data['Forest_2025_r']   = data['Forest_2025'].round(2)

print(f'{len(data)} countries loaded')
data.head(3)

In [ ]:
# Symmetric range so white = 0
abs_max = max(abs(data['Forest_change'].min()), abs(data['Forest_change'].max()))

fig = px.choropleth(
    data,
    locations='Code',
    color='Forest_change',
    hover_name='Entity',
    hover_data={
        'Code':           False,
        'Forest_change':  False,
        'Forest_1990_r':  True,
        'Forest_2025_r':  True,
        'Forest_change_r':True,
    },
    labels={
        'Forest_1990_r':  'Forest 1990 (%)',
        'Forest_2025_r':  'Forest 2025 (%)',
        'Forest_change_r':'Change (pp)',
    },
    # Diverging Red–White–Green, centred at 0
    color_continuous_scale=[
        [0.00, '#d73027'], [0.20, '#f46d43'],
        [0.35, '#fdae61'], [0.45, '#fee08b'],
        [0.50, '#ffffff'],
        [0.55, '#d9ef8b'], [0.65, '#a6d96a'],
        [0.80, '#66bd63'], [1.00, '#1a9850'],
    ],
    range_color=[-abs_max, abs_max],
    title='Forest Cover Change 1990–2025 (percentage points)',
)

fig.update_layout(
    title=dict(x=0.5, xanchor='center', font=dict(size=18)),
    coloraxis_colorbar=dict(title='Change (pp)', tickformat='.1f', len=0.6),
    geo=dict(
        showframe=False,
        showcoastlines=True, coastlinecolor='#888888',
        showland=True,       landcolor='#f0f0f0',
        showocean=True,      oceancolor='#cce5ff',
        projection_type='natural earth',
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=600,
)

fig.show()

In [ ]:
# Save as standalone HTML (include_plotlyjs='cdn' keeps file small ~27 KB)
# For fully offline embed use include_plotlyjs=True (~3 MB)
fig.write_html(str(OUT), full_html=True, include_plotlyjs='cdn')
print(f'Saved to {OUT}  ({OUT.stat().st_size // 1024} KB)')